# Realized Vol Forecasting using HAR-RV

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
sns.set_style("whitegrid")

In [2]:
swaps = pd.read_parquet("../data/processed/usdcweth_swaps.parquet")

## Real Mid Price

In [3]:
df = swaps[['block_timestamp', 'sqrtPriceX96']].dropna().copy()
df['block_timestamp'] = pd.to_datetime(df['block_timestamp'], utc=True)
df = df.sort_values('block_timestamp')

In [4]:
USDC_DEC = 6
WETH_DEC = 18
Q192 = 2 ** 192

df = swaps.sort_values("block_timestamp").copy()
sqrtP = df["sqrtPriceX96"].astype(float)
df["price_weth_per_usdc"] = (sqrtP**2 / Q192)*(10 ** (USDC_DEC - WETH_DEC))
df["price_usdc_per_weth"] = 1/df["price_weth_per_usdc"]

## Intraday log returns

In [5]:
df = df.set_index('block_timestamp')
price_5m = df['price_usdc_per_weth'].resample('5T').last().ffill()

/var/folders/3t/0nrchrfd79xcg6j3q6yk110m0000gn/T/ipykernel_22040/2035167510.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  price_5m = df['price_usdc_per_weth'].resample('5T').last().ffill()
/var/folders/3t/0nrchrfd79xcg6j3q6yk110m0000gn/T/ipykernel_22040/2035167510.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  price_5m = df['price_usdc_per_weth'].resample('5T').last().ffill()


In [6]:
log_price= np.log(price_5m)
rets_5m  = log_price.diff().dropna()

## Realized variance 

In [7]:
rv_daily = (rets_5m**2).groupby(rets_5m.index.date).sum()

In [8]:
rv_daily = (
    rv_daily.to_frame(name='RV').assign(date=lambda x: pd.to_datetime(x.index)).set_index('date')
)

In [9]:
print(rv_daily.head())

                  RV
date                
2021-05-05  0.001869
2021-05-06  0.003099
2021-05-07  0.000687
2021-05-08  0.001765
2021-05-09  0.001354


In [10]:
har = rv_daily.copy()

har['RV_d'] = har['RV']
har['RV_w'] = har['RV'].rolling(window=5, min_periods=5).mean()
har['RV_m'] = har['RV'].rolling(window=22, min_periods=22).mean()

# Target is the next days RV
har['RV_t_plus_1'] = har['RV'].shift(-1)

In [11]:
har = har.dropna()

In [12]:
har.head()

,RV,RV_d,RV_w,RV_m,RV_t_plus_1
date,,,,,
2021-05-26,0.003965,0.003965,0.012028,0.010540,0.002380
2021-05-27,0.002380,0.002380,0.009981,0.010563,0.004347
2021-05-28,0.004347,0.004347,0.006251,0.010620,0.003798
2021-05-29,0.003798,0.003798,0.004658,0.010761,0.002891
2021-05-30,0.002891,0.002891,0.003476,0.010812,0.002001


## Now lets run the model

In [13]:
X = har[['RV_d', 'RV_w', 'RV_m']]
X = sm.add_constant(X)
y = har["RV_t_plus_1"]

split = int(len(har) * 0.7)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [14]:
model = sm.OLS(y_train, X_train).fit()

In [15]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            RV_t_plus_1   R-squared:                       0.341
Model:                            OLS   Adj. R-squared:                  0.337
Method:                 Least Squares   F-statistic:                     84.20
Date:                Sun, 30 Nov 2025   Prob (F-statistic):           6.23e-44
Time:                        08:22:11   Log-Likelihood:                 2439.0
No. Observations:                 493   AIC:                            -4870.
Df Residuals:                     489   BIC:                            -4853.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0005      0.000      4.431      0.0

In [16]:
y_pred = model.predict(X_test)

In [17]:
y_naive = har['RV_d'].shift(0).iloc[split:]
mse_har = mean_squared_error(y_test, y_pred)
mse_naive = mean_squared_error(y_test, y_naive)

print("MSE HAR:", mse_har)
print("MSE naive:", mse_naive)
print("Relative improvement:", 1 - mse_har/mse_naive)

MSE HAR: 1.4522483724983634e-06
MSE naive: 1.420107249934144e-06
Relative improvement: -0.0226328839358505


## Interpretations:
- Respectable R^2 considering that RV is very noisy
- RV_d (0.48, p=0) - Yesterdays variance has strong predictive power, reinforcing idea that vol is persistent.
- RV_w (0.20, p=0.004) - Weekly component also adds significant predictive power. Seems that weekly conditions also matter for forecasting.
- RV_m (0.04, p=0.47) - Monthly component is not significant here.

Conclusion:
It looks like out of sample MSE is actually slightly worse than the naive option of comparing tomorrows RV versus todays RV.

Next Steps:
- Drop monthly
- Model log RV instead of RV
- Maybe a model with more robust se (heteroskedasticity-robust standard errors)

In [18]:
df.head()

,block_number,tx_hash,log_index,tx_index,sender,recipient,amount0,amount1,sqrtPriceX96,liquidity,tick,price_weth_per_usdc,price_usdc_per_weth
block_timestamp,,,,,,,,,,,,,
2021-05-05 01:56:23,12371376,ce7c3c307d820785caa12938012372fc9366a614a6aacf...,26,20,0xE592427A0AEce92De3Edee1F18E0157C05861564,0x5Eefc9306f11a824762CcDaedaC41049EFc7fcc8,-329608,100000000000000,1377932816571815120446551350158799,4303369674465501,195285.0,0.000302,3306.001763
2021-05-05 08:23:26,12373132,9a1c51b0bffbf840948f3b6e3f3e495ba1cd3fa64854f9...,192,172,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xE592427A0AEce92De3Edee1F18E0157C05861564,-164694492,50000000000000000,1378850591292581266780357299649652,4303369674465501,195298.0,0.000303,3301.602222
2021-05-05 09:50:51,12373520,c58715c62a5bf70a6ca09f0e51546d6cad76c8d4fff036...,8,18,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xFB36693ac2DBE8DFC9f89dD0de6015c6Ea66b0BF,-329169,100000000000000,1378852426842022799073024911548633,4303369674465501,195298.0,0.000303,3301.593431
2021-05-05 11:59:57,12374077,288c21b8b4fbf449b1d086a06e43b124ac2bc088c3f536...,86,102,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xFB36693ac2DBE8DFC9f89dD0de6015c6Ea66b0BF,2,-329169,1378852426842016741051966412054516,4304946248093346,195298.0,0.000303,3301.593431
2021-05-05 12:56:56,12374320,67502d8ba373287f6d301f6baa77c5a5f4c80d0753c34c...,257,115,0xE592427A0AEce92De3Edee1F18E0157C05861564,0x22F9dCF4647084d6C31b2765F6910cd85C178C18,1559137299,-467880854065813753,1370241555019945317645788135487819,4304946248093346,195173.0,0.000299,3343.219561


In [ ]:
df.to_csv("../data/engineered/